
# Probleme CSP : ISA + ROA = TELO
Variables  : I, S, A, R, O, E, L, T  (8 lettres distinctes)
Domaine    : {0, 1, ..., 9}
Contraintes:
- Toutes les variables ont des valeurs distinctes
- I != 0  (ISA  est un nombre a 3 chiffres)
- R != 0  (ROA  est un nombre a 3 chiffres)
- T != 0  (TELO est un nombre a 4 chiffres)
- ISA + ROA == TELO


In [1]:
import time
from itertools import permutations

In [2]:

VARIABLES    = ['I', 'S', 'A', 'R', 'O', 'E', 'L', 'T']
DOMAIN       = list(range(10))
LEADING_ZERO = {'I', 'R', 'T'}   # lettres qui ne peuvent pas valoir 0


def word_value(assignment, letters, weights):
    """Calcule la valeur numerique d'un mot a partir de l'assignation."""
    return sum(assignment[l] * w for l, w in zip(letters, weights))



# Methode 1 : Generate and Test (GT)
def generate_and_test():
    solutions = []

    for perm in permutations(DOMAIN, len(VARIABLES)):
        assignment = dict(zip(VARIABLES, perm))

        # Contraintes unaires : premiers chiffres non nuls
        if assignment['I'] == 0 or assignment['R'] == 0 or assignment['T'] == 0:
            continue

        I, S, A = assignment['I'], assignment['S'], assignment['A']
        R, O    = assignment['R'], assignment['O']
        E, L, T = assignment['E'], assignment['L'], assignment['T']

        ISA  = I * 100 + S * 10 + A
        ROA  = R * 100 + O * 10 + A
        TELO = T * 1000 + E * 100 + L * 10 + O

        if ISA + ROA == TELO:
            solutions.append(assignment.copy())

    return solutions

# Methode 2 : Backtracking (BT)

def is_consistent(var, value, assignment):
    """
    Verifie la coherence de l'assignation partielle apres avoir pose var=value.
    Trois niveaux de verification :
      1. Unicite des valeurs (AllDiff)
      2. Contrainte unaire (pas de zero pour I, R, T)
      3. Contrainte globale si toutes les variables sont assignees
    """
    # Niveau 1 : unicite
    if value in assignment.values():
        return False

    # Niveau 2 : contrainte unaire
    if var in LEADING_ZERO and value == 0:
        return False

    # Niveau 3 : contrainte globale complete
    test = assignment.copy()
    test[var] = value

    if len(test) == len(VARIABLES):
        I, S, A = test['I'], test['S'], test['A']
        R, O    = test['R'], test['O']
        E, L, T = test['E'], test['L'], test['T']

        ISA  = I * 100 + S * 10 + A
        ROA  = R * 100 + O * 10 + A
        TELO = T * 1000 + E * 100 + L * 10 + O

        return ISA + ROA == TELO

    return True


def backtrack(assignment, solutions):
    if len(assignment) == len(VARIABLES):
        solutions.append(assignment.copy())
        return

    var = next(v for v in VARIABLES if v not in assignment)

    for value in DOMAIN:
        if is_consistent(var, value, assignment):
            assignment[var] = value
            backtrack(assignment, solutions)
            del assignment[var]


def backtracking():
    solutions = []
    backtrack({}, solutions)
    return solutions




In [3]:
# ---------------------------------------------------------------------------
# Affichage
# ---------------------------------------------------------------------------

def format_solution(sol):
    I, S, A = sol['I'], sol['S'], sol['A']
    R, O    = sol['R'], sol['O']
    E, L, T = sol['E'], sol['L'], sol['T']

    ISA  = I * 100 + S * 10 + A
    ROA  = R * 100 + O * 10 + A
    TELO = T * 1000 + E * 100 + L * 10 + O

    mapping = '  '.join(f"{k}={v}" for k, v in sol.items())
    return f"[ {mapping} ]  =>  {ISA} + {ROA} = {TELO}"


def run_method(name, func):
    print(f"\n{'='*60}")
    print(f" Methode : {name}")
    print(f"{'='*60}")
    start     = time.perf_counter()
    solutions = func()
    elapsed   = time.perf_counter() - start

    print(f" Temps d'execution     : {elapsed:.4f} secondes")
    print(f" Solutions trouvees    : {len(solutions)}")
    for sol in solutions:
        print(f"   {format_solution(sol)}")

    return elapsed, solutions


# ---------------------------------------------------------------------------
# Point d'entree
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    print("Probleme CSP : ISA + ROA = TELO")
    print(f"Variables    : {VARIABLES}")
    print(f"Domaine      : chiffres 0 a 9, tous distincts")
    print(f"Contraintes  : I != 0, R != 0, T != 0")
    print(f"Note         : A est partage par ISA et ROA")
    print(f"               O est partage par ROA et TELO")

    t_gt, sol_gt = run_method("Generate and Test (GT)", generate_and_test)
    t_bt, sol_bt = run_method("Backtracking       (BT)", backtracking)

    print(f"\n{'='*60}")
    print(" Comparaison des performances")
    print(f"{'='*60}")
    print(f" GT  : {t_gt:.4f} s")
    print(f" BT  : {t_bt:.4f} s")

    if t_bt < t_gt:
        ratio = t_gt / t_bt if t_bt > 0 else float('inf')
        print(f" BT est {ratio:.1f}x plus rapide que GT")
    else:
        ratio = t_bt / t_gt if t_gt > 0 else float('inf')
        print(f" GT est {ratio:.1f}x plus rapide que BT")

    meme = sorted([str(s) for s in sol_gt]) == sorted([str(s) for s in sol_bt])
    print(f"\n Les deux methodes trouvent les memes solutions : {meme}")

Probleme CSP : ISA + ROA = TELO
Variables    : ['I', 'S', 'A', 'R', 'O', 'E', 'L', 'T']
Domaine      : chiffres 0 a 9, tous distincts
Contraintes  : I != 0, R != 0, T != 0
Note         : A est partage par ISA et ROA
               O est partage par ROA et TELO

 Methode : Generate and Test (GT)
 Temps d'execution     : 1.7497 secondes
 Solutions trouvees    : 104
   [ I=2  S=4  A=9  R=7  O=8  E=0  L=3  T=1 ]  =>  249 + 789 = 1038
   [ I=2  S=5  A=4  R=7  O=8  E=0  L=3  T=1 ]  =>  254 + 784 = 1038
   [ I=2  S=5  A=9  R=7  O=8  E=0  L=4  T=1 ]  =>  259 + 789 = 1048
   [ I=2  S=6  A=9  R=7  O=8  E=0  L=5  T=1 ]  =>  269 + 789 = 1058
   [ I=2  S=8  A=3  R=7  O=6  E=0  L=4  T=1 ]  =>  283 + 763 = 1046
   [ I=2  S=9  A=3  R=7  O=6  E=0  L=5  T=1 ]  =>  293 + 763 = 1056
   [ I=3  S=0  A=7  R=9  O=4  E=2  L=5  T=1 ]  =>  307 + 947 = 1254
   [ I=3  S=0  A=8  R=9  O=6  E=2  L=7  T=1 ]  =>  308 + 968 = 1276
   [ I=3  S=2  A=8  R=7  O=6  E=0  L=9  T=1 ]  =>  328 + 768 = 1096
   [ I=3  S=5  A=2  R=